In [6]:
import csv

with open("orders.csv", encoding="utf-8-sig") as f:
    rows=list(csv.DictReader(f))

In [7]:
# База
#1
def to_num(s):
    s=str(s).strip().replace(" ","").replace(",", ".")
    if s=="":
        return 0
    return float(s)

def clean_city(s):
    s=s.strip().lower()
    cities={"almaty":"Алматы", "astana":"Астана", "shymkent":"Шымкент", "karaganda":"Караганда", "aktobe":"Актобе", "taraz":"Тараз"}
    if s in cities:
        return cities[s]
    return s.title()

# Check actual CSV values and add alternative spellings if necessary
print("Cities:",sorted({r["city"] for r in rows}))
print("Statuses:",sorted({r["status"] for r in rows}))
print("Categories:",sorted({r["category"] for r in rows}))

# Paid / cancelled statuses are checked using the dataset's actual values
paid_values={"paid", "оплачен", "оплачено", "true", "1"}
cancel_values={"cancelled", "canceled", "отменен", "отменён", "отменено"}

def is_paid(r):
    return r["status"].strip().lower() in paid_values

def is_cancelled(r):
    return r["status"].strip().lower() in cancel_values

#2
category_count={}
for r in rows:
    category=r["category"]
    category_count[category]=category_count.get(category,0)+1

sorted_categories=sorted(category_count, key=category_count.get, reverse=True)
print("Orders by category:")
for category in sorted_categories:
    print(category,category_count[category])

#3
category_revenue={}
category_paid_count={}
for r in rows:
    if is_paid(r):
        category=r["category"]
        amount=to_num(r["amount"])
        category_revenue[category]=category_revenue.get(category,0)+amount
        category_paid_count[category]=category_paid_count.get(category,0)+1

print("Category | Revenue | Average paid order")
for category in sorted_categories:
    revenue=category_revenue.get(category,0)
    count=category_paid_count.get(category,0)
    average=0
    if count>0:
        average=revenue/count
    print(f"{category} | {revenue:,.0f} ₸ | {average:,.1f} ₸")

#4
month_count={}
for r in rows:
    month=r["order_date"][:7]
    month_count[month]=month_count.get(month,0)+1
print("Orders by month:")
for month in sorted(month_count):
    print(month,month_count[month])


Cities: [' Алматы', 'Almaty', 'Astana ', 'Shymkent', 'АЛМАТЫ', 'Актобе', 'Алматы', 'Астана', 'Атырау', 'Караганда', 'Шымкент', 'алматы', 'астана', 'шымкент']
Statuses: ['оплачен', 'отменён']
Categories: ['Дом', 'Книги', 'Красота', 'Одежда', 'Электроника']
Orders by category:
Одежда 1529
Электроника 1185
Дом 1105
Красота 991
Книги 653
Category | Revenue | Average paid order
Одежда | 18,321,080 ₸ | 13,285.8 ₸
Электроника | 65,270,240 ₸ | 61,286.6 ₸
Дом | 10,516,320 ₸ | 10,654.8 ₸
Красота | 5,507,620 ₸ | 6,188.3 ₸
Книги | 2,294,100 ₸ | 3,935.0 ₸
Orders by month:
2026-09 2782
2026-10 2681


In [8]:
# Стандарт
#5
month_revenue={}
for r in rows:
    if is_paid(r):
        month=r["order_date"][:7]
        month_revenue[month]=month_revenue.get(month,0)+to_num(r["amount"])

for month in sorted(month_revenue):
    print(month,f"{month_revenue[month]:,.0f} ₸")

sep=month_revenue.get("2026-09",0)
oct=month_revenue.get("2026-10",0)
change=0
if sep!=0:
    change=(oct-sep)/sep*100
print(f"October vs September: {change:.1f}%")

#6
city_orders={}
city_cancelled={}
for r in rows:
    city=clean_city(r["city"])
    city_orders[city]=city_orders.get(city,0)+1
    if is_cancelled(r):
        city_cancelled[city]=city_cancelled.get(city,0)+1
for city in sorted(city_orders):
    share=city_cancelled.get(city,0)/city_orders[city]*100
    print(f"{city}: {share:.1f}% cancelled")

#7
customer_revenue={}
for r in rows:
    if is_paid(r):
        customer=r["customer_id"]
        customer_revenue[customer]=customer_revenue.get(customer,0)+to_num(r["amount"])

top_customers=sorted(customer_revenue,key=customer_revenue.get,reverse=True)
print("Top 5 customers:")
for customer in top_customers[:5]:
    print(customer,f"{customer_revenue[customer]:,.0f} ₸")
if len(top_customers)>0:
    print("Top customer:",top_customers[0])

#8
customers_by_city={}
for r in rows:
    city=clean_city(r["city"])
    if city not in customers_by_city:
        customers_by_city[city]=set()
    customers_by_city[city].add(r["customer_id"])
for city in sorted(customers_by_city):
    print(city,len(customers_by_city[city]),"unique customers")

2026-09 57,489,850 ₸
2026-10 44,419,510 ₸
October vs September: -22.7%
Актобе: 10.3% cancelled
Алматы: 7.9% cancelled
Астана: 8.8% cancelled
Атырау: 7.5% cancelled
Караганда: 8.9% cancelled
Шымкент: 20.4% cancelled
Top 5 customers:
TEST 29,999,970 ₸
C01627 257,180 ₸
C00485 215,920 ₸
C01338 192,080 ₸
C02122 191,670 ₸
Top customer: TEST
Актобе 358 unique customers
Алматы 1351 unique customers
Астана 981 unique customers
Атырау 138 unique customers
Караганда 451 unique customers
Шымкент 655 unique customers


In [9]:
# Челлендж
#9
seen=set()
unique_rows=[]
duplicates_by_month={}
duplicates=0
for r in rows:
    row_tuple=tuple(r.values())
    if row_tuple in seen:
        duplicates+=1
        month=r["order_date"][:7]
        duplicates_by_month[month]=duplicates_by_month.get(month,0)+1
    else:
        seen.add(row_tuple)
        unique_rows.append(r)
print("Full duplicates:",duplicates)
for month in sorted(duplicates_by_month):
    print(month,duplicates_by_month[month])

#10
clean_month_revenue={}
for r in unique_rows:
    if is_paid(r) and r["customer_id"]!="TEST":
        month=r["order_date"][:7]
        clean_month_revenue[month]=clean_month_revenue.get(month,0)+to_num(r["amount"])
for month in sorted(clean_month_revenue):
    print(month,f"{clean_month_revenue[month]:,.0f} ₸")

clean_sep=clean_month_revenue.get("2026-09",0)
clean_oct=clean_month_revenue.get("2026-10",0)
clean_change=0
if clean_sep!=0:
    clean_change=(clean_oct-clean_sep)/clean_sep*100
print(f"Before cleaning: {change:.1f}%")
print(f"After cleaning: {clean_change:.1f}%")

#11
customer_orders={}
for r in unique_rows:
    customer=r["customer_id"]
    customer_orders[customer]=customer_orders.get(customer,0)+1
sorted_customers=sorted(customer_orders,key=customer_orders.get,reverse=True)
if len(sorted_customers)>0:
    top_customer=sorted_customers[0]
    print("Most orders:",top_customer,customer_orders[top_customer])


Full duplicates: 180
2026-09 180
2026-09 35,243,410 ₸
2026-10 34,419,520 ₸
Before cleaning: -22.7%
After cleaning: -2.3%
Most orders: C01818 10
